# Brain Tumor Segmentation — Data Exploration

This notebook loads and explores the BraTS2020 dataset before any training takes place. The goal is to understand how multi-modal 3D MRI data is structured, verify that image-mask pairs are complete and aligned, and get a feel for tumor size and shape variation across patients — all of which will shape decisions in the preprocessing and training notebooks that follow.

## What this notebook does

1. Extracts the BraTS2020 dataset from the zip file
2. Verifies folder structure and checks that each patient has all 4 MRI modalities + a segmentation mask
3. Loads sample volumes with `nibabel` and visualizes 2D slices across modalities
4. Computes dataset-wide statistics (volume dimensions, tumor size distribution, class balance)

## Dataset

- **BraTS2020** (Training + Validation), source: [Kaggle — awsaf49/brats20-dataset-training-validation](https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation)
- 369 patients (training set), 4 MRI modalities per patient (T1, T1ce, T2, FLAIR), NIfTI format (`.nii`)
- Labels: background, necrotic/non-enhancing tumor core (NCR/NET), peritumoral edema (ED), GD-enhancing tumor (ET)
- License: CC0 Public Domain

## Output

- `results/figures/sample_slices.png`
- `results/figures/modality_comparison.png`
- `results/figures/tumor_size_distribution.png`

In [1]:
# Import libraries

import os
import zipfile
import random

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\brain-tumor-segmentation-3d\repo"

PATHS = {
    # Raw zip file
    "brats_zip": os.path.join(BASE_DIR, "data", "raw", "brats20-nifti.zip"),

    # Extracted dataset folder
    "brats_dir": os.path.join(BASE_DIR, "data", "brats2020"),

    # Output folders
    "figures": os.path.join(BASE_DIR, "results", "figures"),
    "metrics": os.path.join(BASE_DIR, "results", "metrics"),
}

for key in ["figures", "metrics"]:
    os.makedirs(PATHS[key], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:12s} -> {path}")